In [1]:
import os
os.chdir('application')

In [2]:
from ingest.pdf_with_tables import extract_pdf_content

In [3]:
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [4]:
pdf_pages = extract_pdf_content('../assets/CP-Cuti-dan-Izin-Khusus.pdf')

In [6]:
pdf_pages[1]

{'page': 2,
 'narrative_text': '● REMOTE AREA adalah area atau lokasi kerja yang ditetapkan Manajemen sebagai\nlokasi yang jauh dari pemukiman warga atau tidak memiliki transportasi umum yang\nmemadai. 17\n18\n● SITE adalah kantor cabang dari Business Unit.\n● PERJALANAN DINAS adalah perjalanan yang dilakukan oleh karyawan ke luar lokasi\nkerja dalam rangka pelaksanaan tugas atau kepentingan Perusahaan. 19\n●\nSURAT PERINTAH PERJALANAN DINAS (SPPD) adalah surat digital atau non-digital\nyang dibuat dan disetujui Atasan sebagai bentuk pemberian tugas bagi karyawan untuk\n20\nmelakukan Perjalanan Dinas.\nCUTI TAHUNAN\n● Diberikan Kepada:\n○ 21\nKaryawan Kontrak: Diberikan 1 hari kerja setiap melewati 1 bulan masa kerja.\n○ Karyawan Tetap: Diberikan per Kuartal. Setelah bekerja 3 bulan berturut-turut,\n22\nhak cuti akan muncul sebanyak 3 hari kerja.\n● 23\nJumlah Kuota: 15 hari.\n● Jumlah Maksimal Dalam 1 Pengajuan: 12 hari. 24\n● Pengajuan dan Penggunaan Cuti Tahunan:\n○ 25\nDapat dilaku

In [5]:
# 1. Initialize the Hugging Face Embedding Model for Indonesian
def get_huggingface_embeddings():
    """Initializes and returns a Hugging Face embedding model for Indonesian."""
    model_name = "LazarusNLP/all-indo-e5-small-v4"
    model_kwargs = {'device': 'cpu'}  # Use 'cuda' for GPU
    encode_kwargs = {'normalize_embeddings': True}
    
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )
    return embeddings

In [ ]:
# !pip install hf_xet

  Using cached hf_xet-1.1.10-cp37-abi3-win_amd64.whl.metadata (4.7 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   -------------------------- ------------- 1.8/2.8 MB 5.6 MB/s eta 0:00:01
   ------------------------------------- -- 2.6/2.8 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 5.1 MB/s  0:00:00


In [7]:
embeddings = get_huggingface_embeddings()

C:\Users\USER\AppData\Local\Temp\ipykernel_9824\3133064784.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Can't load the model for 'LazarusNLP/all-indo-e5-small-v4'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'LazarusNLP/all-indo-e5-small-v4' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

In [ ]:
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="percentile", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.95 # Adjust this value to control chunk size
)

In [ ]:
import os
from langchain_experimental.text_splitter import SemanticChunker






# 2. Instantiate the Semantic Chunker
# The `breakpoint_threshold_type` tells the chunker how to find the topic changes.
# "percentile" is a great default. It means we'll split at the 95th percentile
# of similarity differences. In simple terms, it splits where the topic change
# is most significant.
text_splitter = SemanticChunker(
    embeddings=embeddings, 
    breakpoint_threshold_type="percentile", # Other options: "standard_deviation", "interquartile", "gradient"
    breakpoint_threshold_amount=0.95 # Adjust this value to control chunk size
)


# 3. Prepare an example text with a clear topic shift
# The text starts by talking about Jakarta's history and then shifts to modern transportation.
jakarta_text = (
    "Jakarta, secara resmi bernama Daerah Khusus Ibukota Jakarta, adalah ibu kota negara dan kota terbesar di Indonesia. "
    "Terletak di pesisir bagian barat laut Pulau Jawa, kota ini telah menjadi pusat ekonomi, budaya, dan politik selama berabad-abad. "
    "Sejarahnya dimulai pada abad ke-4 sebagai sebuah pelabuhan perdagangan Hindu bernama Sunda Kelapa. "
    "Pada tahun 1527, Fatahillah dari Kesultanan Demak menaklukkan pelabuhan tersebut dan menamainya Jayakarta. "
    # --- The semantic chunker should identify a topic shift here ---
    "Sistem transportasi modern di Jakarta sangat bergantung pada bus dan kereta api. "
    "TransJakarta adalah sistem bus rapid transit (BRT) yang menjadi tulang punggung transportasi umum. "
    "Selain itu, MRT Jakarta dan LRT Jakarta juga telah beroperasi untuk mengurangi kemacetan lalu lintas yang parah di kota ini."
)


# 4. Create the chunks
docs = text_splitter.create_documents([jakarta_text])


# 5. View the results
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print("\n" + "="*20 + "\n")